In [ ]:
import pandas as pd
import numpy as np
import re, string, warnings
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RepeatedKFold
from sklearn.metrics import mean_absolute_error
from sklearn.decomposition import PCA
from sentence_transformers import SentenceTransformer
from sklearn.ensemble import StackingRegressor
from scipy.sparse import hstack
from sklearn.neural_network import MLPRegressor
import lightgbm as lgb
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
warnings.filterwarnings('ignore')


In [ ]:
# =============================
# 🧹 Data Cleaning
# =============================
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\d+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [ ]:
# =============================
# 📥 Load Train Data
# =============================
data = pd.read_csv('Data_without_imagelink.csv')
data['clean_text'] = data['catalog_content'].fillna('').apply(clean_text)

In [ ]:
# =============================
# 🧠 Text Embeddings
# =============================
print("🔹 Generating sentence embeddings...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embedding_model.encode(data['clean_text'], show_progress_bar=True)

In [ ]:
# =============================
# 🔍 TF-IDF + PCA Compression
# =============================
tfidf = TfidfVectorizer(stop_words=stopwords.words('english'), max_features=5000)
tfidf_matrix = tfidf.fit_transform(data['clean_text'])
pca = PCA(n_components=50, random_state=42)
tfidf_pca = pca.fit_transform(tfidf_matrix.toarray())

In [ ]:
# =============================
# 💡 Combine Features
# =============================
X_full = np.hstack([embeddings, tfidf_pca])
y = data['price'].fillna(0)


In [ ]:
# =============================
# 🔢 Outlier Handling
# =============================
q_low, q_high = y.quantile(0.01), y.quantile(0.99)
mask = (y >= q_low) & (y <= q_high)
X_full, y = X_full[mask], y[mask]


In [ ]:
# =============================
# 🔁 Log Transformation
# =============================
y_log = np.log1p(y)


In [ ]:
# =============================
# ⚙ Scaling
# =============================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_full)


In [ ]:
# =============================
# 🎯 Only Fold 1 Train/Val Split
# =============================
rkf = RepeatedKFold(n_splits=5, n_repeats=2, random_state=42)
train_idx, val_idx = next(iter(rkf.split(X_scaled)))  # ✅ TAKE ONLY FIRST SPLIT

X_train, X_val = X_scaled[train_idx], X_scaled[val_idx]
y_train, y_val = y_log.iloc[train_idx], y_log.iloc[val_idx]

In [ ]:
# =============================
# 🤖 Define Models (with LGBM ✅)
# =============================
xgb = XGBRegressor(n_estimators=600, learning_rate=0.05, max_depth=8,
                   subsample=0.8, colsample_bytree=0.8, reg_lambda=1, n_jobs=-1, random_state=42)

lgbm = lgb.LGBMRegressor(n_estimators=600, learning_rate=0.05, num_leaves=64,
                         feature_fraction=0.8, bagging_fraction=0.8, random_state=42)

cat = CatBoostRegressor(iterations=800, learning_rate=0.05, depth=7,
                        random_state=42, verbose=0)

mlp = MLPRegressor(hidden_layer_sizes=(256, 128, 64), activation='relu',
                   solver='adam', learning_rate_init=0.001, max_iter=600,
                   random_state=42, early_stopping=True)

stack_model = StackingRegressor(
    estimators=[('xgb', xgb), ('lgbm', lgbm), ('cat', cat)],
    final_estimator=mlp,
    n_jobs=-1
)



In [ ]:
# =============================
# 🚀 Train on Fold 1 Only
# =============================
print("\n🔥 Training on Fold 1 only...")
stack_model.fit(X_train, y_train)


In [ ]:
# =============================
# 📥 Load Test Data and Apply Same Processing
# =============================
test_df = pd.read_csv('test.csv')
test_df['clean_text'] = test_df['catalog_content'].fillna('').apply(clean_text)

# Embeddings
test_embeddings = embedding_model.encode(test_df['clean_text'], show_progress_bar=True)

# TF-IDF + PCA (use same fitted objects)
test_tfidf = tfidf.transform(test_df['clean_text'])
test_tfidf_pca = pca.transform(test_tfidf.toarray())

# Combine
X_test = np.hstack([test_embeddings, test_tfidf_pca])
X_test_scaled = scaler.transform(X_test)

# Predict
test_preds = np.expm1(stack_model.predict(X_test_scaled))

In [ ]:
# =============================
# 💾 Save Output as CSV
# =============================
output = pd.DataFrame({
    'sample_id': test_df['sample_id'],
    'price': test_preds
})

output.to_csv('test_out.csv', index=False)
print("\n✅ test_out.csv GENERATED SUCCESSFULLY!")